<a href="https://colab.research.google.com/github/iThaysCambi/FormacaoDados_GenerationBrasil/blob/main/Atividade_Hotel_Limpeza_Visualizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Dashboard Interativo - Análise Hoteleira (Plotly)
# Este notebook realiza o tratamento de dados de uma base de hotelaria e gera visuais dinâmicos e interativos usando a biblioteca **Plotly**.

### Etapas executadas:
# 1. Upload dinâmico do ficheiro `hotel_dataset_messy.csv`.
# 2. Filtro de idades (apenas hóspedes entre 18 e 100 anos).
# 3. Correção de valores de faturação negativos para absolutos.
# 4. Cálculo da métrica de permanência (`Total_Nights`).
# 5. Construção de uma grelha interativa de subplots (1 linha x 3 colunas).


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Carregar os dados
df_hotel = pd.read_csv("/content/hotel_dataset_messy.csv")
print('Informações da base de dados')
df_hotel.info()


Informações da base de dados
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Guest_Name          1500 non-null   object 
 1   Age                 1500 non-null   int64  
 2   Room_Type           1500 non-null   object 
 3   CheckIn_Date        1500 non-null   object 
 4   CheckOut_Date       1500 non-null   object 
 5   Booking_Channel     1500 non-null   object 
 6   Total_Charges       1500 non-null   float64
 7   Satisfaction_Score  1366 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 93.9+ KB


In [ ]:
import io
import pandas as pd
from google.colab import files

print("Por favor, seleciona o ficheiro 'hotel_dataset_messy.csv' do teu computador:")
# Criar o botão de upload do Colab
uploaded = files.upload()

# Ler o conteúdo do ficheiro diretamente da memória da sessão
df = pd.read_csv(io.BytesIO(uploaded["hotel_dataset_messy.csv"]))
print("\nFicheiro carregado com sucesso!")

# Exibir as primeiras linhas para validação rápida
df.head()

Por favor, seleciona o ficheiro 'hotel_dataset_messy.csv' do teu computador:


Saving hotel_dataset_messy.csv to hotel_dataset_messy.csv

Ficheiro carregado com sucesso!


,Guest_Name,Age,Room_Type,CheckIn_Date,CheckOut_Date,Booking_Channel,Total_Charges,Satisfaction_Score
0,pAulo chiavini,69,Standard,2025-08-21,2025-08-23,Site Oficial,296.967888,1.0
1,aNa pereira,32,Standard,2025-08-28,2025-09-10,Booking.com,2235.383385,5.0
2,luYz silva,78,Standard,2025-01-04,2025-01-18,Site Oficial,1907.702878,4.0
3,luYz silva,38,Deluxe,2025-06-30,2025-07-14,Booking.com,3751.333568,2.0
4,aNa silva,41,Deluxe,2025-02-28,2025-03-02,Booking.com,575.659560,3.0


In [ ]:
# Converter colunas de data para o formato datetime correto do Pandas
df["CheckIn_Date"] = pd.to_datetime(df["CheckIn_Date"])
df["CheckOut_Date"] = pd.to_datetime(df["CheckOut_Date"])

# 1. Limpar Idades Bizarras (Apenas maiores de 18 e menores ou igual a 100 anos)
df_filtrado = df[(df["Age"] >= 18) & (df["Age"] <= 100)].copy()

# 2. Corrigir Faturamento (Transformar valores negativos em positivos usando valor absoluto)
df_filtrado["Total_Charges"] = df_filtrado["Total_Charges"].abs()

# 3. Criar uma Nova Métrica (Diferença de dias entre check-out e check-in)
df_filtrado["Total_Nights"] = (
    df_filtrado["CheckOut_Date"] - df_filtrado["CheckIn_Date"]
).dt.days

# Agrupar a faturação por canal de reserva para alimentar o gráfico de barras
faturamento_por_canal = (
    df_filtrado.groupby("Booking_Channel")["Total_Charges"]
    .sum()
    .reset_index()
)

print("Tratamento de dados concluído com sucesso!")

Tratamento de dados concluído com sucesso!


In [14]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Criar a estrutura do painel: 1 linha e 3 colunas
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Distribuição de Idade dos Hóspedes",
        "Faturamento Total por Canal",
        "Distribuição de Noites por Hospedagem",
    ),
    horizontal_spacing=0.08,  # Espaçamento lateral entre os gráficos
)

# --- Subplot 1: Histograma de Idades ---
fig.add_trace(
    go.Histogram(
        x=df_filtrado["Age"],
        nbinsx=15,
        name="Idade",
        marker_color="#5dade2",
        marker=dict(line=dict(color="black", width=1)),
    ),
    row=1,
    col=1,
)

# --- Subplot 2: Gráfico de Barras de Faturamento ---
fig.add_trace(
    go.Bar(
        x=faturamento_por_canal["Booking_Channel"],
        y=faturamento_por_canal["Total_Charges"],
        name="Faturamento",
        marker_color="#e67e22",
        marker=dict(line=dict(color="black", width=1)),
    ),
    row=1,
    col=2,
)

# --- Subplot 3: Histograma de Noites Estendidas ---
fig.add_trace(
    go.Histogram(
        x=df_filtrado["Total_Nights"],
        nbinsx=10,
        name="Noites",
        marker_color="#58d68d",
        marker=dict(line=dict(color="black", width=1)),
    ),
    row=1,
    col=3,
)

# --- Customização Geral do Layout ---
fig.update_layout(
    title_text="Dashboard Interativo de Análise Hoteleira",
    title_x=0.5,  # Centraliza o título principal do dashboard
    title_font=dict(size=20),
    width=1300,  # Largura ideal para caber no ecrã do Colab sem quebrar
    height=500,  # Altura dos gráficos
    showlegend=False,  # Remove legendas duplicadas para limpar o visual
    template="plotly_white",  # Tema corporativo com fundo limpo
)

# Nomear os eixos de cada gráfico de forma individual
fig.update_xaxes(title_text="Idade", row=1, col=1)
fig.update_yaxes(title_text="Frequência", row=1, col=1)

fig.update_xaxes(title_text="Canal de Reserva", row=1, col=2)
fig.update_yaxes(title_text="Faturamento ($)", row=1, col=2)

fig.update_xaxes(title_text="Quantidade de Noites", row=1, col=3)
fig.update_yaxes(title_text="Frequência", row=1, col=3)

# Exibir os gráficos na tela (permite zoom e análise ao passar o rato)
fig.show()

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Preparar dados específicos para os novos gráficos
# Contagem para o gráfico de pizza
quartos_distribuição = (
    df_filtrado["Room_Type"].value_counts().reset_index()
)
quartos_distribuição.columns = ["Room_Type", "Contagem"]

# --- CONFIGURAÇÃO DO PLOTLY (GRADE DE 2 LINHAS X 3 COLUNAS) ---
# Definimos os tipos de gráficos de cada célula. O de pizza precisa do tipo 'domain'
fig = make_subplots(
    rows=2,
    cols=3,
    specs=[
        [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],  # Linha 1
        [
            {"type": "domain"},
            {"type": "xy", "colspan": 2},
            None,
        ],  # Linha 2 (Pizza + Dispersão ocupando 2 colunas)
    ],
    subplot_titles=(
        "Distribuição de Idade dos Hóspedes",
        "Faturamento Total por Canal",
        "Distribuição de Noites por Hospedagem",
        "Distribuição por Tipo de Quarto",
        "Relação: Faturamento vs. Nota de Satisfação",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.15,  # Espaço entre a linha 1 e a linha 2
)

# ================= LINHA 1 =================

# Subplot 1: Histograma de Idades
fig.add_trace(
    go.Histogram(
        x=df_filtrado["Age"],
        nbinsx=15,
        name="Idade",
        marker_color="#5dade2",
        marker=dict(line=dict(color="black", width=1)),
    ),
    row=1,
    col=1,
)

# Subplot 2: Gráfico de Barras de Faturamento
fig.add_trace(
    go.Bar(
        x=faturamento_por_canal["Booking_Channel"],
        y=faturamento_por_canal["Total_Charges"],
        name="Faturamento",
        marker_color="#e67e22",
        marker=dict(line=dict(color="black", width=1)),
    ),
    row=1,
    col=2,
)

# Subplot 3: Histograma de Noites
fig.add_trace(
    go.Histogram(
        x=df_filtrado["Total_Nights"],
        nbinsx=10,
        name="Noites",
        marker_color="#58d68d",
        marker=dict(line=dict(color="black", width=1)),
    ),
    row=1,
    col=3,
)

# ================= LINHA 2 =================

# Subplot 4: Gráfico de Pizza (Estilo Donut para visual mais moderno)
fig.add_trace(
    go.Pie(
        labels=quartos_distribuição["Room_Type"],
        values=quartos_distribuição["Contagem"],
        name="Quartos",
        hole=0.4,  # Transforma a pizza em rosca/donut
        marker=dict(colors=["#9b59b6", "#34495e", "#f1c40f"]),
    ),
    row=2,
    col=1,
)

# Subplot 5: Gráfico de Dispersão (Ocupa as colunas 2 e 3 da linha 2)
fig.add_trace(
    go.Scatter(
        x=df_filtrado["Total_Charges"],
        y=df_filtrado["Satisfaction_Score"],
        mode="markers",
        name="Satisfação",
        marker=dict(
            size=9,
            color=df_filtrado[
                "Satisfaction_Score"
            ],  # Muda a cor do ponto baseado na nota
            colorscale="Viridis",  # Escala de cor bonita e legível
            showscale=True,  # Mostra a barra de escala de cor do lado
            colorbar=dict(
                title="Nota", x=1.02, len=0.4, y=0.25
            , opacity=0.7),
            line=dict(width=1, color="DarkSlateGrey"),
        ),
    ),
    row=2,
    col=2,
)


# --- CUSTOMIZAÇÃO GERAL DO LAYOUT ---
fig.update_layout(
    title_text="Dashboard Executivo de Análise Hoteleira (Pleno)",
    title_x=0.5,
    title_font=dict(size=22),
    width=1350,  # Mantém uma boa largura para a grade
    height=900,  # Aumentamos a altura para acomodar as duas linhas perfeitamente
    showlegend=True,  # Ativado para ajudar a identificar as fatias da pizza
    template="plotly_white",
)

# Nomear os eixos (Linha 1)
fig.update_xaxes(title_text="Idade", row=1, col=1)
fig.update_yaxes(title_text="Frequência", row=1, col=1)

fig.update_xaxes(title_text="Canal de Reserva", row=1, col=2)
fig.update_yaxes(title_text="Faturamento ($)", row=1, col=2)

fig.update_xaxes(title_text="Quantidade de Noites", row=1, col=3)
fig.update_yaxes(title_text="Frequência", row=1, col=3)

# Nomear os eixos (Linha 2 - O gráfico de pizza não usa eixos X/Y)
fig.update_xaxes(title_text="Faturamento Total ($)", row=2, col=2)
fig.update_yaxes(title_text="Nota de Satisfação", row=2, col=2)

# Exibir o dashboard completo
fig.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.scatter.marker.ColorBar: 'opacity'

Did you mean "xpad"?

    Valid properties:
        bgcolor
            Sets the color of padded area.
        bordercolor
            Sets the axis line color.
        borderwidth
            Sets the width (in px) or the border enclosing this
            color bar.
        dtick
            Sets the step in-between ticks on this axis. Use with
            `tick0`. Must be a positive number, or special strings
            available to "log" and "date" axes. If the axis `type`
            is "log", then ticks are set every 10^(n*dtick) where n
            is the tick number. For example, to set a tick mark at
            1, 10, 100, 1000, ... set dtick to 1. To set tick marks
            at 1, 100, 10000, ... set dtick to 2. To set tick marks
            at 1, 5, 25, 125, 625, 3125, ... set dtick to
            log_10(5), or 0.69897000433. "log" has several special
          